# Module 2: Single Agent

Build a Decision Intelligence agent with tools, run it, and inspect the agentic loop in action. By the end of this module you will have a working agent that can look up company data, market benchmarks, and competitor information: and you will understand exactly what happens inside the loop at each step.

**Prerequisites:** Python 3.10+, AWS credentials configured (Strands uses Amazon Bedrock by default)

> **This workshop uses Strands Agents**: the open-source agent SDK. The same patterns carry over to other agent frameworks.

## Tools Used in This Module

| Tool | What it does | Key behavior |
|------|-------------|--------------|
| `get_company_data(company_name)` | Returns MarketNest's financial and operational data | Looks up by lowercased name; returns CLV, churn rate, revenue, top-spender segment |
| `get_market_benchmarks(industry)` | Returns industry benchmark data for e-commerce | Returns avg CLV, churn, subscription adoption rates, CLV lift range |
| `get_competitor_data(competitor_name)` | Returns a competitor's premium tier details | Returns pricing, pilot approach, adoption rate, CLV lift, time to profitability |

All three tools use mock data: no external APIs or keys needed. Swap them for real data sources to ground the agent in live market intelligence.

In [ ]:
# Install dependencies (takes ~20 seconds on first run)
%pip install -r requirements.txt

---

## How to select your preffered choice of model

In [ ]:
from strands import Agent
from strands.models import BedrockModel

model = BedrockModel(model_id="us.anthropic.claude-opus-5")

---

## Create a Single agent Recommendation Engine

Define the System Prompt

In [ ]:
SYSTEM_PROMPT = '''

You are a senior Corporate Strategy Advisor.

You will receive a Situation Summary. Produce an Executive Strategy Briefing with:

1) Recommendation (1-2 lines)
2) Options considered (A/B/C)
3) Tradeoffs (bullets: gain vs risk)
4) Risks + mitigations (bullets)
5) Assumptions + unknowns (bullets)
6) Confidence (0-100) + why (3 bullets)
7) Escalate? (Yes/No + reason)
8) Next steps (owner + timeline bullets)

Situation:
{Scenario}

Rules:
- Do not invent metrics. If missing, write "Unknown".
- Make tradeoffs explicit: "We gain X; we risk Y."
- If constraints are unmet or approvals missing, set Escalate=Yes.
- Keep sections in the same order.

'''

---

Create the Agent using Model and System Prompt

In [ ]:
from strands import Agent

agent = Agent(
    system_prompt=SYSTEM_PROMPT,
    model=model
)

---

Run the Agent for input Situation Summary

In [ ]:
SITUATION_SUMMARY = '''

Situation Summary: We are considering launching a Premium Subscription Tier for MarketNest's top-performing product line. The subscription would bundle free next-day delivery, a members-only rewards program, and concierge customer support.

Customer segment & value delivered: 
- Repeat buyers in metro zones (highest order frequency and basket size). 
- Value delivered is faster fulfillment, member-only pricing, and early product access — designed to deepen loyalty and lift purchase frequency.

Target dates / Target milestones: 
- Decision needed within 2 weeks. 
- Pilot launch targeted for Q3.
- first membership cohort onboarded by end of Q3.
- mid-pilot review at week 6; go/no-go on full rollout at month 3.

Success Criteria: 
- Increase checkout conversion by 10% in metro zones within 3 months. 
- Current average order value (AOV) is approximately $86 (secondary watch metric).

Rollout with exit criteria: 
- Phased rollout starting with a 5% metro-zone cohort, expanding to 25% only after the mid-pilot review clears. 
- Exit criteria — halt and roll back if checkout conversion lift is under 3% by week 6, if next-day delivery is met on fewer than 95% of member orders, or if cost-to-serve per subscriber exceeds incremental margin.

Questions: 
- Can our fulfillment network sustain 95%+ next-day delivery in metro zones at pilot volume? 
- What subscription price point maximizes enrollment without eroding incremental margin? 
- Should we anchor on net-new subscribers or convert existing loyalty members first? 
- Which metro zones show the conversion headroom to hit the 10% lift?

Stakeholders: Chief Revenue Officer, Head of Growth Marketing, Supply Chain Lead, Customer Experience Director..

The options are: 
- Launch a paid Premium tier in metro zones anchored on the top 5 products in Q1.
- Launch a lower-priced "Plus" tier (priority shipping only, no discounts) across a broader SKU set to maximize enrollment.
- Convert existing top loyalty-program members into Premium subscribers before any net-new acquisition.

'''

result = agent(SITUATION_SUMMARY)

---

Observe the key operational metrics

In [ ]:
# How many messages and tokens did the full brief take?
summary = result.metrics.get_summary()
usage = summary.get("accumulated_usage", {})
print("Single-agent on full brief: metrics:")
print(f"  Cycles (LLM calls):  {summary.get('total_cycles', 'n/a')}")
print(f"  Input tokens:        {usage.get('inputTokens', 'n/a')}")
print(f"  Output tokens:       {usage.get('outputTokens', 'n/a')}")
print(f"  Messages in context: {len(agent.messages)}")
print()
print("One agent played: Researcher + Option Analyst x3 + Synthesizer.")
print("As complexity grows the context bloats and the model loses focus.")
print("That is the ceiling. Module 3 shows how to break past it.")

---

# Optional

---

## Part 1: Create and Run the Agent

Wire the tools into an `Agent` with a system prompt. The LLM uses both the system prompt and the tool docstrings to decide what to do at each step.

In [ ]:
SYSTEM_PROMPT = '''You are a Decision Intelligence Analyst for a technology company.
You help business leaders gather data and context before making strategic decisions.
Use your available tools to look up company data, market benchmarks, and competitor information.

Guidelines:
- Always use tools to answer questions: never guess when real data is available.
- Be concise and data-driven.
- Surface the most relevant numbers for the decision at hand.'''

agent = Agent(
    system_prompt=SYSTEM_PROMPT,
    model=model
)

# The agent decides which tools to call based on the request and tool docstrings.
result = agent(
    "What is MarketNest's current CLV and churn rate? "
    "How does that compare to the e-commerce industry benchmark?"
)

---

## Part 2: Inspect the Agent Loop

The agentic loop is:

```
User message → LLM reasons → selects a tool → tool executes → result appended to context → LLM reasons again → ... repeat until done
```

Every step is stored in `agent.messages`. Let's see what actually happened.

In [ ]:
import json as _json

print(f"Total messages in conversation: {len(agent.messages)}")
print("=" * 65)

for i, msg in enumerate(agent.messages):
    role = msg["role"]
    content = msg.get("content", [])

    if role == "user":
        # content can be a string or a list of content blocks
        if isinstance(content, str):
            text = content[:100]
        elif content:
            text = str(content[0].get("text", content[0]))[:100]
        else:
            text = ""
        print(f"\n[{i}] USER:         {text}")

    elif role == "assistant":
        for block in content:
            if "text" in block:
                print(f"\n[{i}] ASSISTANT:    {block['text'][:120]}...")
            elif "toolUse" in block:
                tu = block["toolUse"]
                inp = _json.dumps(tu.get("input", {}))
                print(f"\n[{i}] TOOL CALL:    {tu['name']}({inp})")

    elif role == "tool":
        for block in content:
            result_text = str(block.get("content", ""))[:120]
            print(f"\n[{i}] TOOL RESULT:  {result_text}...")

print("\n" + "=" * 65)

In [ ]:
# AgentResult carries usage metrics for the last invocation
summary = result.metrics.get_summary()
usage = summary.get("accumulated_usage", {})
print("Loop metrics:")
print(f"  Cycles (LLM calls):  {summary.get('total_cycles', 'n/a')}")
print(f"  Input tokens:        {usage.get('inputTokens', 'n/a')}")
print(f"  Output tokens:       {usage.get('outputTokens', 'n/a')}")

---

## Part 3: Try It Yourself

The agent keeps its conversation history across cells: each call to `agent(...)` adds to the same context window. Try the prompts below in order to see multi-turn behavior.

In [ ]:
# The agent keeps conversation history across cells, each call adds to the same context.
# Ask a follow-up using the MarketNest context already in memory:
agent("What did ShopMart do when they launched their premium tier? How did it perform?")

In [ ]:
agent("And PrimeStore? Which approach: ShopMart's or PrimeStore's: had better ROI?")

In [ ]:
# Uncomment to explore further:
# agent("What percentage of MarketNest's top spenders said they'd pay for a premium tier?")
# agent("Based on industry benchmarks, how long should MarketNest expect before subscription profit?")

In [ ]:
# ── Lens 2: result.metrics.get_summary() ─────────────────────────────────
# Zero extra config — token usage, cycle count, and per-tool stats
# are on every AgentResult your agent call returns.
import json as _json

summary = result.metrics.get_summary()

print("=== Loop metrics ===")
print(f"  total_cycles:     {summary.get('total_cycles')}")
print(f"  total_duration_s: {round(summary.get('total_duration', 0), 2)}")
print()

usage = summary.get("accumulated_usage", {})
print("=== Token usage ===")
print(f"  input_tokens:  {usage.get('inputTokens', 'n/a')}")
print(f"  output_tokens: {usage.get('outputTokens', 'n/a')}")
print(f"  total_tokens:  {usage.get('totalTokens', 'n/a')}")
print()

tool_usage = summary.get("tool_usage", {})
if tool_usage:
    print("=== Per-tool stats ===")
    for tool_name, data in tool_usage.items():
        stats = data.get("execution_stats", {})
        print(
            f"  {tool_name}: "
            f"calls={stats.get('call_count', 0)} | "
            f"success={stats.get('success_count', 0)} | "
            f"avg_time={round(stats.get('average_time', 0), 3)}s"
        )
else:
    print("(no tool_usage in summary: the agent answered from memory)")

---

## Part 4: The Ceiling: One Agent, One Giant Context

So far the agent handled focused questions well. Now let's give it the **full Decision Brief**: three options, competitive context, financial targets, timeline: and ask for a complete recommendation.

Watch what happens: the agent tries to do everything in a single context window.

In [ ]:
DECISION_BRIEF = '''
DECISION BRIEF: MarketNest Premium Tier Launch

Company: MarketNest (2M active users, mid-size e-commerce)
Decision owners: VP Product + CFO approval required

Context:
- Current annual churn rate: 22%  |  Industry avg: 25%
- Customer Lifetime Value (CLV): $340  |  Industry avg: $290
- A major competitor launched a similar tier last quarter
- Q1 survey: 38% of top-spending users expressed interest in a premium tier

Options to evaluate:
  Option A: Exclusive Premium: invite-only for top 10% of spenders, $19.99/mo
  Option B: Gradual Rollout: 5% A/B test pilot with kill-switch, $14.99/mo
  Option C: Full Launch: open to all users immediately, $12.99/mo + 30-day free trial

For each option provide:
  - Pros and cons
  - Implementation complexity (Low / Medium / High)
  - Top 3 risks with mitigations
  - Revenue and CLV impact estimate

Then recommend the best option with justification.

Success target: +15% CLV improvement within 6 months
Budget: $2M  |  Decision deadline: 2027-01-31
'''

# Fresh agent with no prior context, clean baseline
ceiling_agent = Agent(
    system_prompt=SYSTEM_PROMPT,
    model=model
)

result_ceiling = ceiling_agent(DECISION_BRIEF)

In [ ]:
# How many messages and tokens did the full brief take?
ceiling_summary = result_ceiling.metrics.get_summary()
ceiling_usage = ceiling_summary.get("accumulated_usage", {})
print("Single-agent on full brief: metrics:")
print(f"  Cycles (LLM calls):  {ceiling_summary.get('total_cycles', 'n/a')}")
print(f"  Input tokens:        {ceiling_usage.get('inputTokens', 'n/a')}")
print(f"  Output tokens:       {ceiling_usage.get('outputTokens', 'n/a')}")
print(f"  Messages in context: {len(ceiling_agent.messages)}")
print()
print("One agent played: Researcher + Option Analyst x3 + Synthesizer.")
print("As complexity grows the context bloats and the model loses focus.")
print("That is the ceiling. Module 3 shows how to break past it.")

---

## Key Takeaways

| Concept | What you saw |
|---------|-------------|
| `@tool` | Python function the LLM can call: docstring is the routing logic |
| `Agent(tools=[], system_prompt=...)` | Assembles the agent: model + tools + instructions |
| `agent.messages` | Full conversation history: every user turn, LLM response, tool call, tool result |
| `result.metrics` | Loop execution stats: cycles and tokens consumed |
| **The ceiling** | One agent juggling researcher + analyst + synthesizer roles bloats the context |

---

## What's Next

In **Module 3: Sequential Chain**, you break this into a clean pipeline: each stage passes its output to the next as a focused handoff. Same task, three specialized agents, predictable and debuggable flow.

---

## Want a real multi-turn conversation?

Each notebook cell is a single turn. To chat back and forth with the agent maintaining memory across turns, run the companion script in a terminal:

```bash
cd samples/02-single-agent
uv pip install -r requirements.txt
uv run python chat.py
```

Type your messages, `quit` or Ctrl+C to exit.